# Aufgabe 3 – Eigenes neuronales Netz (Schachbrett)

Netz mit 5 Neuronen von Grund auf, Lernalgorithmus, Fehler-/Gewichtsverlauf.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt


## 3a) Aufbau überlegen (Neuronen, Verbindungen, optimale Gewichte)

Aufbau des Netzes: Für die Erkennung eines 2×2-Schachbretts werden fünf Neuronen benötigt: vier Input-Neuronen (je eines pro Feld) und ein Output-Neuron. Ein Hidden-Layer existiert nicht — die vier Inputs sind direkt mit dem Output verbunden. Es gibt also vier Verbindungen mit je einem Gewicht $w_1,\dots,w_4$.

Formen von Input und Output: Der Input ist ein Vektor aus vier Werten $[x_1,x_2,x_3,x_4]$ mit $x_i \in \{0,1\}$ (schwarz/weiß), die das 2×2-Feld zeilenweise kodieren. Ein „1"-Feld regt sein Input-Neuron mit $I_\mathrm{max}$ zum Feuern an, ein „0"-Feld hält es bei $I_0$ (Ruhe). Der Output ist ein Neuron: Feuert es, wird das Muster als Schachbrett erkannt (`True`), sonst nicht (`False`). Der Strom ins Output-Neuron ergibt sich nach Gl. (14) als gewichtete Summe der Spannungen $I_\mathrm{out} = \sum_i w_i\,U_i$.

Optimale Gewichte: Um das Zielmuster (schwarze Diagonale, z. B. $[1,0,0,1]$) von den anderen abzugrenzen, müssen die Gewichte auf den beiden schwarzen Diagonalfeldern hoch und auf den übrigen null sein, also z. B. $w = [1,0,0,1]$. Dann treiben nur die zwei gemeinsam feuernden Diagonal-Neuronen das Output über die Schwelle, während einzelne Felder oder die andere Diagonale es nicht schaffen. Die absolute Größe der Gewichte ist dabei nebensächlich, entscheidend ist das Verhältnis (Diagonale stark, Rest schwach); Skalierungen wie $[0.5,0,0,0.5]$ oder $[2,0,0,2]$ liefern dasselbe Ergebnis.

Wieso nur ein Diagonalmuster erkannt werden kann: Die Wichtungsfaktoren haben die Einheit einer elektrischen Leitfähigkeit und sind daher physikalisch positiv ($w_i \ge 0$). Bei ausschließlich positiven Gewichten kann ein zusätzliches aktives (schwarzes) Feld den Strom ins Output-Neuron nur erhöhen, nie senken:
$$I_\mathrm{out}(1,1,1,1) = I_\mathrm{out}(1,0,0,1) + \underbrace{w_2\,U_2^{\text{feuernd}} + w_3\,U_3^{\text{feuernd}}}_{\ge 0}\,.$$
Der Antrieb von $[1,1,1,1]$ ist also stets $\ge$ dem von $[1,0,0,1]$. Feuert das Zielmuster, muss $[1,1,1,1]$ erst recht feuern. Das Netz kann die Diagonalfelder belohnen, die übrigen Felder aber nicht bestrafen, dafür bräuchte es negative Gewichte (eine „Gegenstimme"), die als negative Leitfähigkeit physikalisch nicht existieren.
Die Verwendung negativer Gewichte entgegen der physikalischen Realität führen zu falschen Ergebnissen, da diese mit negativen Spannungen Muster als falsch positiv erkennt. 

Konsequenz: Das Netz erkennt zuverlässig „schwarze Diagonale", kann aber die drei Muster $[1,0,1,1]$, $[1,1,0,1]$ und $[1,1,1,1]$ (Diagonale schwarz plus Extra-Felder) nicht ausschließen. Damit ist das erreichbare Optimum 13 von 16 korrekt, keine Schwäche der Implementierung, sondern die prinzipielle Grenze eines einlagigen Netzes mit positiven Gewichten. Ein echtes Schachbrett (Diagonale schwarz und die anderen Felder weiß) sowie beide Schachbretter zugleich erfordern eine versteckte Schicht.

## 3b) NN mit 5 Neuronen (I_0 als Minimum)

In dieser Zelle wird das Modul network importiert und die Funktion predict an zwei Beispielmustern getestet. predict setzt das 2x2 Muster in Eingangsstroeme um (schwarzes Feld auf I_MAX, weisses Feld auf I_0), löst für jedes der vier Input-Neuronen und für das eine Output-Neuron das Hodgkin-Huxley-Modell und prüft, ob das Output-Neuron feuert. Über die Funktion clamp_current wird sichergestellt, dass der Strom in einem Neuron den Minimalwert I_0 nicht unterschreitet, wie in der Aufgabe gefordert.

Das Ergebnis bestätigt, dass das Netz aus fünf Neuronen technisch funktioniert. Das Zielmuster (1,0,0,1) löst ein Feuern des Output-Neurons aus (Ausgabe True), während das leere Feld (0,0,0,0) kein Feuern erzeugt (Ausgabe False). Damit ist gezeigt, dass die Grundmechanik richtig arbeitet. Ob das Netz auch zuverlaessig zwischen Schachbrett und Nichtschachbrett trennt, wird erst in Aufgabe 3c über alle 16 Muster geprüft.

In [ ]:
from src import network as net
import numpy as np

# Schnelltest: läuft predict und liefert True/False?
w = np.array([1.0, -1.0, -1.0, 1.0])
print("Zielmuster [1,0,0,1]:", net.predict(w, np.array([1, 0, 0, 1])))
print("Leeres Feld [0,0,0,0]:", net.predict(w, np.array([0, 0, 0, 0])))

## 3c) Alle Gewichte = 1 vs. optimale Gewichte

In dieser Zelle wird ueber alle 16 moeglichen Muster geprueft, wie gut das Netz mit einem bestimmten Gewichtssatz zwischen Schachbrett und Nichtschachbrett unterscheidet. Die Funktion teste_gewichte ruft fuer jedes Muster predict auf, vergleicht die Ausgabe mit dem Sollwert (nur (1,0,0,1) gilt als Schachbrett) und zaehlt die korrekten Treffer. Geprueft werden zwei Gewichtssaetze: alle Gewichte gleich 1 und der optimale positive Satz mit hohen Gewichten auf der Diagonale.

Mit gleichen Gewichten erkennt das Netz das Schachbrett nicht zuverlaessig. Da alle Felder gleich stark eingehen, feuert das Output-Neuron vor allem nach der Anzahl aktiver Felder und trennt die Klassen kaum. Mit dem optimalen positiven Satz (1,0,0,1) steigt die Zahl der korrekten Muster deutlich auf 13 von 16. Die verbleibenden drei Fehler treten immer bei den Mustern (1,0,1,1), (1,1,0,1) und (1,1,1,1) auf, also genau dann, wenn die Zieldiagonale schwarz ist und zusaetzlich weitere Felder aktiv sind. Die Antwort auf die Frage, ob das Ergebnis immer stimmt, lautet somit nein. Auch mit optimalen Gewichten bleiben diese drei Muster falsch, weil positive Gewichte die anderen Felder nicht bestrafen koennen. Das ist die in Aufgabe 3a beschriebene prinzipielle Grenze und motiviert den Lernalgorithmus in 3d.

In [ ]:
import itertools

# alle 16 möglichen 2x2-Muster
muster = list(itertools.product([0, 1], repeat=4))

def ist_schachbrett(p):
    return p == (1, 0, 0, 1)      # dein Zielmuster (ggf. anpassen)

def teste_gewichte(weights):
    weights = np.array(weights, dtype=float)
    korrekt = 0
    for p in muster:
        vorhersage = net.predict(weights, np.array(p))
        soll = ist_schachbrett(p)
        if vorhersage == soll:
            korrekt += 1
            markierung = ""
        else:
            markierung = "   <-- falsch"
        print(f"{p}   Vorhersage: {str(vorhersage):5}   Soll: {str(soll):5}{markierung}")
    print(f"\n{korrekt} von {len(muster)} korrekt\n")

# 3c, Teil 1: alle Gewichte = 1
print("=== Alle Gewichte = 1 ===")
teste_gewichte([1, 1, 1, 1])

# 3c, Teil 2: optimale Gewichte
print("=== Optimale Gewichte ===")
teste_gewichte([1, 0, 0, 1])

## 3d) Maschinelles Lernen (Gewichte je Pruefergebnis anpassen)

In dieser Zelle wird der Lernalgorithmus ausgefuehrt. Zunaechst werden alle 16 Muster erzeugt und das Zielmuster mehrfach hinzugefuegt, damit die seltene Schachbrettklasse im Training ausreichend oft vorkommt (Balancing). Die Funktion train startet mit zufaelligen positiven Gewichten aus dem Bereich (0,1] und geht die Muster mehrere Durchgaenge lang durch. Fuer jedes Muster vergleicht sie die Vorhersage mit dem Sollwert und passt die Gewichte der aktiven Felder in Richtung des Fehlers an. Sollte das Netz feuern, tat es aber nicht, werden die Gewichte erhoeht, im umgekehrten Fall gesenkt. Nach jeder Anpassung werden die Gewichte auf Werte groesser oder gleich null geklemmt, damit sie physikalisch gueltige Leitfaehigkeiten bleiben. Fehlerzahl und Gewichte werden je Durchgang protokolliert.

Der gelernte Gewichtssatz laeuft von selbst auf die in 3a hergeleitete Struktur zu, also hohe Gewichte auf den beiden Diagonalfeldern und Werte nahe null auf den uebrigen. Das Netz findet damit eigenstaendig die beste positive Loesung, ohne dass die optimalen Gewichte vorgegeben werden.

In [ ]:
pats = list(itertools.product([0, 1], repeat=4))
ziel = (1, 0, 0, 1)

# Balancing: Zielmuster mehrfach zeigen, damit das Netz nicht "immer nein" lernt
patterns = pats + [ziel] * 6
targets  = [1 if p == ziel else 0 for p in pats] + [1] * 6

# Achtung: Training ruft predict sehr oft auf -> dauert einige Minuten.
# Fuer schnellere Laeufe epochs verringern oder groeberes t uebergeben.
w, fehler_verlauf, gewichte_verlauf = net.train(
    patterns, targets, learning_rate=0.2, epochs=25, seed=1)

print("Gelernte Gewichte:", np.round(w, 2))
print("Fehler pro Durchgang:", list(fehler_verlauf))

## 3e) Fehler und Gewichte ueber Durchgaenge grafisch

In dieser Zelle werden die im Training protokollierten Groessen grafisch dargestellt. Der linke Plot zeigt die Anzahl der Fehler pro Durchgang, der rechte die Entwicklung der vier Gewichte ueber die Durchgaenge.

Der Fehlerverlauf sinkt in den ersten Durchgaengen und stabilisiert sich anschliessend bei einem kleinen Wert groesser als null. Das Netz wird also nicht vollstaendig fehlerfrei, sondern erreicht sein Optimum von 13 von 16 korrekt erkannten Mustern. Das deckt sich mit der Grenze aus Aufgabe 3a, denn die drei Muster mit schwarzer Diagonale und zusaetzlichen aktiven Feldern lassen sich mit positiven Gewichten nicht abtrennen. Der Gewichtsverlauf zeigt, wie die beiden Diagonalgewichte hoch bleiben, waehrend die anderen beiden gegen null laufen. Wiederholt man den Lernvorgang mit verschiedenen Startwerten, landet er reproduzierbar bei derselben Struktur, was fuer die Robustheit des Algorithmus spricht.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Fehler ueber die Durchgaenge
ax1.plot(fehler_verlauf, marker="o")
ax1.set_xlabel("Durchgang (Epoche)")
ax1.set_ylabel("Anzahl Fehler")
ax1.set_title("Fehler ueber die Durchgaenge")
ax1.grid(alpha=0.3)

# Entwicklung der vier Gewichte
for i in range(4):
    ax2.plot(gewichte_verlauf[:, i], label=f"w{i+1}")
ax2.set_xlabel("Durchgang (Epoche)")
ax2.set_ylabel("Gewicht")
ax2.set_title("Entwicklung der Gewichte")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Robustheit (3e): mehrmals mit verschiedenen seeds trainieren und vergleichen.
# Achtung: dauert entsprechend laenger.
# for s in range(3):
#     w_s, f_s, _ = net.train(patterns, targets, learning_rate=0.2, epochs=25, seed=s)
#     print(f"seed {s}: Endfehler {f_s[-1]}, Gewichte {np.round(w_s,2)}")

## 3f) (Zusatz) Beide Schachbrettmuster erkennen